# Model Training Notebook
This notebook trains FastText embeddings, fits a classifier, and evaluates results.



In [ ]:
pip install -U scikit-learn==1.7.2


In [ ]:
!pip install gensim scikit-learn nltk numpy pandas joblib

## Imports & Constants
Load required libraries and set constants (stopwords, etc.).

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from gensim.models import FastText
from nltk.corpus import stopwords
import nltk
import joblib
nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))

## Data Loading
Read the raw dataset and perform initial selection of relevant columns.

In [ ]:
df = pd.read_csv("drugsComTrain_raw.csv")

In [ ]:
df = df[['drugName', 'condition', 'review', 'rating']].dropna()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.iloc[15]["review"]

In [ ]:
df = df[(df["rating"] >= 7)]

In [ ]:
df =df.dropna()

In [ ]:
df

## Preprocessing
Text cleaning and stopword removal; create `clean_review`.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<*.?>','',text) 
    text = re.sub(r'[^a-z\s]','',text)
    text = " ".join([w for w in text.split() if w not in STOPWORDS])
    return text

In [ ]:
df['clean_review'] = df['review'].apply(clean_text)

In [ ]:
df['condition'] = df['condition'].astype(str)

In [ ]:
conditions_count = df['condition'].value_counts()
valid_conditions = conditions_count[conditions_count>4].index
df = df[df['condition'].isin(valid_conditions)].reset_index(drop=True)

In [ ]:
sentences = [row.split() for row in df['clean_review']]

## FastText Training
Train FastText on cleaned reviews to produce word embeddings.

In [ ]:
print("FastText model is training, please wait⌛")
ft_model = FastText(
    sentences = sentences,
    vector_size = 100,
    window = 5,
    min_count = 2,
    sg = 1,
    epochs = 10
)

## Feature Extraction
Convert cleaned reviews to fixed-size vectors using the trained embeddings.

In [ ]:
def get_vector(text):
    words = [w for w in text.split() if w in ft_model.wv]
    if not words:
        return np.zeros(ft_model.vector_size)
    return np.mean([ft_model.wv[w] for w in words], axis = 0)

X = np.vstack(df['clean_review'].apply(get_vector).values)

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df['condition'])


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y 
)

## Classifier Training
Fit a classifier (Logistic Regression) on the extracted features.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
print("\n✅ Accuracy:", accuracy_score(y_test, y_pred))
print("\n📋 Classification Report:\n", classification_report(y_test, y_pred,zero_division=1, target_names=le.classes_))

## Save Models
Persist the trained embeddings, classifier, and label encoder for later use.

In [ ]:
# FastText embeddings
ft_model.save("fasttext_drug_review.model")

# Logistic Regression classifier
import joblib
joblib.dump(model, "condition_classifier.pkl")

# Label encoder
joblib.dump(le, "label_encoder.pkl")


## Inference Example
Load saved models and run a sample prediction to verify behavior.

In [ ]:
# ==========================
# 📦 Import Required Libraries
# ==========================
import pandas as pd
import numpy as np
import joblib
from gensim.models import FastText

# ==========================
# 🧠 Load Saved Models
# ==========================
ft_model = FastText.load("fasttext_drug_review.model")
model = joblib.load("condition_classifier.pkl")
le = joblib.load("condition_label_encoder.pkl")

# ==========================
# 🔁 Define Utility Function
# ==========================
def get_vector(text):
    words = text.lower().split()
    words = [w for w in words if w in ft_model.wv]
    if not words:
        return np.zeros(ft_model.vector_size)
    return np.mean([ft_model.wv[w] for w in words], axis=0)

def predict_condition(review):
    vec = get_vector(review).reshape(1, -1)
    pred = model.predict(vec)
    return le.inverse_transform(pred)[0]

# ==========================
# 💬 Example Prediction
# ==========================
sample_symptom = "Erectile Dysfunction"
predicted_condition = predict_condition(sample_symptom)


print("💬 Symptom:", sample_symptom)
print("🩺 Predicted condition:", predicted_condition)


In [ ]:
import pandas as pd

# Existing code
sample_symptom = "fever"
predicted_condition = predict_condition(sample_symptom)

# Get top 3 drugs associated with the predicted condition
condition_drugs = df[df['condition'] == predicted_condition]
if not condition_drugs.empty:
    top_drugs = condition_drugs['drugName'].value_counts().head(3)
    top_drugs_list = top_drugs.index.tolist()
    top_drugs_counts = top_drugs.values.tolist()
else:
    top_drugs_list = []
    top_drugs_counts = []

# Print results
print("💬 Symptom:", sample_symptom)
print("🩺 Predicted condition:", predicted_condition)
print("💊 Top 3 drugs associated with this condition:")
if top_drugs_list:
    for drug, count in zip(top_drugs_list, top_drugs_counts):
        print(f"  - {drug} ({count} reviews)")
else:
    print("  - No drugs found for this condition in the dataset.")

In [ ]:
pip install seaborn

## Metrics & Visualizations
Compute performance metrics and generate plots for analysis.

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    matthews_corrcoef,
    precision_recall_fscore_support
)
import seaborn as sns
import matplotlib.pyplot as plt
from gensim.models import FastText
import joblib
from nltk.corpus import stopwords
import nltk

# Download stopwords if not already available
try:
    STOPWORDS = set(stopwords.words('english'))
except:
    nltk.download('stopwords')
    STOPWORDS = set(stopwords.words('english'))

# ==========================================
# LOAD MODELS
# ==========================================
print("Loading models...")
ft_model = FastText.load("fasttext_drug_review.model")
model = joblib.load("condition_classifier.pkl")
le = joblib.load("condition_label_encoder.pkl")
print("✅ Models loaded successfully!\n")

# ==========================================
# LOAD AND PREPROCESS DATA
# ==========================================
print("Loading data...")
df = pd.read_csv("drugsComTrain_raw.csv")
df = df[['drugName', 'condition', 'review', 'rating']].dropna()
df = df[(df["rating"] >= 7)]

# Clean text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>','',text) 
    text = re.sub(r'[^a-z\s]','',text)
    text = " ".join([w for w in text.split() if w not in STOPWORDS])
    return text

df['clean_review'] = df['review'].apply(clean_text)
df['condition'] = df['condition'].astype(str)

# Filter conditions (same as training)
conditions_count = df['condition'].value_counts()
valid_conditions = conditions_count[conditions_count > 4].index
df = df[df['condition'].isin(valid_conditions)].reset_index(drop=True)

# Further filter to only conditions that exist in the label encoder
df = df[df['condition'].isin(le.classes_)].reset_index(drop=True)
print(f"✅ Data loaded: {len(df)} samples, {len(le.classes_)} conditions\n")

# ==========================================
# VECTORIZE DATA
# ==========================================
print("Vectorizing reviews...")
def get_vector(text):
    words = [w for w in text.split() if w in ft_model.wv]
    if not words:
        return np.zeros(ft_model.vector_size)
    return np.mean([ft_model.wv[w] for w in words], axis=0)

X = np.vstack(df['clean_review'].apply(get_vector).values)
y = le.transform(df['condition'])
print("✅ Vectorization complete!\n")

# ==========================================
# SPLIT DATA (same split as training)
# ==========================================
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Test set size: {len(y_test)} samples\n")

# ==========================================
# MAKE PREDICTIONS
# ==========================================
print("Making predictions...")
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)
print("✅ Predictions complete!\n")

# ==========================================
# 1. OVERALL METRICS
# ==========================================
print("=" * 60)
print("📊 OVERALL PERFORMANCE METRICS")
print("=" * 60)

accuracy = accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
kappa = cohen_kappa_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"✅ Accuracy:              {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"📏 Precision (Macro):     {precision_macro:.4f}")
print(f"📏 Precision (Weighted):  {precision_weighted:.4f}")
print(f"🎯 Recall (Macro):        {recall_macro:.4f}")
print(f"🎯 Recall (Weighted):     {recall_weighted:.4f}")
print(f"⚖️  F1-Score (Macro):      {f1_macro:.4f}")
print(f"⚖️  F1-Score (Weighted):   {f1_weighted:.4f}")
print(f"🔗 Cohen's Kappa:         {kappa:.4f}")
print(f"🧮 Matthews Corr Coef:    {mcc:.4f}")

# ==========================================
# 2. DETAILED CLASSIFICATION REPORT
# ==========================================
print("\n" + "=" * 60)
print("📋 DETAILED CLASSIFICATION REPORT (PER CLASS)")
print("=" * 60)
print(classification_report(
    y_test, 
    y_pred, 
    target_names=le.classes_, 
    zero_division=0,
    digits=4
))

# ==========================================
# 3. TOP/BOTTOM PERFORMING CLASSES
# ==========================================
precisions, recalls, f1_scores, supports = precision_recall_fscore_support(
    y_test, y_pred, zero_division=0
)

performance_df = pd.DataFrame({
    'Condition': le.classes_,
    'Precision': precisions,
    'Recall': recalls,
    'F1-Score': f1_scores,
    'Support': supports
}).sort_values('F1-Score', ascending=False)

print("\n" + "=" * 60)
print("🏆 TOP 10 BEST PERFORMING CONDITIONS")
print("=" * 60)
print(performance_df.head(10).to_string(index=False))

print("\n" + "=" * 60)
print("⚠️  TOP 10 WORST PERFORMING CONDITIONS")
print("=" * 60)
print(performance_df.tail(10).to_string(index=False))

# ==========================================
# 4. CONFUSION MATRIX ANALYSIS
# ==========================================
cm = confusion_matrix(y_test, y_pred)

print("\n" + "=" * 60)
print("🔍 CONFUSION MATRIX INSIGHTS")
print("=" * 60)
print(f"Total Predictions: {len(y_test)}")
print(f"Correct Predictions: {np.trace(cm)}")
print(f"Incorrect Predictions: {len(y_test) - np.trace(cm)}")

# ==========================================
# 5. CONFIDENCE ANALYSIS
# ==========================================
max_probas = np.max(y_pred_proba, axis=1)
correct_mask = (y_test == y_pred)

print("\n" + "=" * 60)
print("🎲 PREDICTION CONFIDENCE ANALYSIS")
print("=" * 60)
print(f"Average Confidence (All): {np.mean(max_probas):.4f}")
print(f"Average Confidence (Correct): {np.mean(max_probas[correct_mask]):.4f}")
print(f"Average Confidence (Incorrect): {np.mean(max_probas[~correct_mask]):.4f}")
print(f"Min Confidence: {np.min(max_probas):.4f}")
print(f"Max Confidence: {np.max(max_probas):.4f}")

# Confidence bins
print("\n📊 Confidence Distribution:")
bins = [0, 0.3, 0.5, 0.7, 0.9, 1.0]
labels = ['0-30%', '30-50%', '50-70%', '70-90%', '90-100%']
for i in range(len(bins)-1):
    count = np.sum((max_probas >= bins[i]) & (max_probas < bins[i+1]))
    pct = (count / len(max_probas)) * 100
    print(f"  {labels[i]}: {count:5d} predictions ({pct:5.2f}%)")

# ==========================================
# 6. VISUALIZATION: CONFUSION MATRIX
# ==========================================
print("\n" + "=" * 60)
print("📊 GENERATING VISUALIZATIONS...")
print("=" * 60)

# Only plot if not too many classes
if len(le.classes_) <= 50:
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', 
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    print("✅ Confusion matrix saved as 'confusion_matrix.png'")
else:
    print("⚠️  Too many classes for confusion matrix visualization")

# ==========================================
# 7. VISUALIZATION: F1-SCORES
# ==========================================
plt.figure(figsize=(12, 8))
top_20 = performance_df.head(20)
plt.barh(top_20['Condition'], top_20['F1-Score'], color='steelblue')
plt.xlabel('F1-Score', fontsize=12)
plt.ylabel('Condition', fontsize=12)
plt.title('Top 20 Conditions by F1-Score', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('f1_scores_top20.png', dpi=300, bbox_inches='tight')
print("✅ F1-scores chart saved as 'f1_scores_top20.png'")

# ==========================================
# 8. VISUALIZATION: CONFIDENCE DISTRIBUTION
# ==========================================
plt.figure(figsize=(10, 6))
plt.hist(max_probas[correct_mask], bins=50, alpha=0.7, label='Correct Predictions', color='green')
plt.hist(max_probas[~correct_mask], bins=50, alpha=0.7, label='Incorrect Predictions', color='red')
plt.xlabel('Prediction Confidence', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Prediction Confidence', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=300, bbox_inches='tight')
print("✅ Confidence distribution saved as 'confidence_distribution.png'")

# ==========================================
# 9. VISUALIZATION: PRECISION-RECALL-F1
# ==========================================
plt.figure(figsize=(14, 6))
top_30 = performance_df.head(30)
x = np.arange(len(top_30))
width = 0.25

plt.bar(x - width, top_30['Precision'], width, label='Precision', alpha=0.8)
plt.bar(x, top_30['Recall'], width, label='Recall', alpha=0.8)
plt.bar(x + width, top_30['F1-Score'], width, label='F1-Score', alpha=0.8)

plt.xlabel('Condition', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Precision, Recall, and F1-Score for Top 30 Conditions', fontsize=14, fontweight='bold')
plt.xticks(x, top_30['Condition'], rotation=90, fontsize=8)
plt.legend()
plt.tight_layout()
plt.savefig('precision_recall_f1.png', dpi=300, bbox_inches='tight')
print("✅ Precision-Recall-F1 chart saved as 'precision_recall_f1.png'")

# ==========================================
# 10. SAVE METRICS TO CSV
# ==========================================
performance_df.to_csv('per_class_metrics.csv', index=False)
print("✅ Per-class metrics saved as 'per_class_metrics.csv'")

# Summary metrics
summary_metrics = {
    'Metric': ['Accuracy', 'Precision (Macro)', 'Precision (Weighted)', 
               'Recall (Macro)', 'Recall (Weighted)', 'F1-Score (Macro)', 
               'F1-Score (Weighted)', 'Cohen\'s Kappa', 'Matthews Corr Coef'],
    'Value': [accuracy, precision_macro, precision_weighted, recall_macro, 
              recall_weighted, f1_macro, f1_weighted, kappa, mcc]
}
summary_df = pd.DataFrame(summary_metrics)
summary_df.to_csv('overall_metrics.csv', index=False)
print("✅ Overall metrics saved as 'overall_metrics.csv'")

# ==========================================
# 11. INTERPRETATION GUIDE
# ==========================================
print("\n" + "=" * 60)
print("📖 METRICS INTERPRETATION GUIDE")
print("=" * 60)
print(f"""
🎯 **Accuracy**: Overall correctness (correct predictions / total predictions)
   - Your model: {accuracy*100:.2f}%
   
📏 **Precision**: Of all predicted positives, how many were correct?
   - High precision = Few false positives
   - Macro: {precision_macro*100:.2f}% | Weighted: {precision_weighted*100:.2f}%
   
🎯 **Recall**: Of all actual positives, how many did we find?
   - High recall = Few false negatives
   - Macro: {recall_macro*100:.2f}% | Weighted: {recall_weighted*100:.2f}%
   
⚖️  **F1-Score**: Harmonic mean of precision and recall (balanced metric)
   - Macro: {f1_macro*100:.2f}% | Weighted: {f1_weighted*100:.2f}%
   - Macro = unweighted average (treats all classes equally)
   - Weighted = weighted by class support (better for imbalanced data)
   
🔗 **Cohen's Kappa**: Agreement corrected for chance (0=random, 1=perfect)
   - Your model: {kappa:.4f}
   - <0.20: Poor | 0.21-0.40: Fair | 0.41-0.60: Moderate
   - 0.61-0.80: Good | >0.80: Excellent
   
🧮 **Matthews Corr Coef**: Balanced measure even for imbalanced classes
   - Your model: {mcc:.4f}
   - Range: -1 (worst) to +1 (perfect), 0 = random
""")

print("=" * 60)
print("✨ EVALUATION COMPLETE!")
print("=" * 60)
print(f"\n📁 Generated files:")
print("  - confusion_matrix.png (if applicable)")
print("  - f1_scores_top20.png")
print("  - confidence_distribution.png")
print("  - precision_recall_f1.png")
print("  - per_class_metrics.csv")
print("  - overall_metrics.csv")